# 11 — Expectation Values

Make precise the single number a real experiment reports — the expectation value $\langle A\rangle = \langle\psi|A|\psi\rangle$ — as an *average of measurement outcomes*, size its error bar, and build the bridge to tomography.

**Prerequisites:** `01/08_pauli_gates` (the bracket $\langle\psi|A|\psi\rangle$, eigenvalues as outcomes), `01/09_hadamard_and_measurement_bases` (measuring in a basis), `01/04_measurement_and_born_rule` (the Born rule), `01/05_inner_product_and_dirac` (basis expansion).

**What you'll be able to do**
- Prove the **spectral rule** $\langle A\rangle = \sum_k \lambda_k P(k)$: the bracket of `01/08` *is* the average of the eigenvalue outcomes, weighted by their Born probabilities — and check it by hand for $Z$ and $X$.
- Estimate $\langle Z\rangle$ from $N$ shots, derive the error bar $\sqrt{(1 - \langle Z\rangle^2)/N}$, and test it as a prediction against repeated trials.
- Estimate all three Pauli expectations of one state from shots — the entire recipe of tomography (`01/16`).

A single measurement returns a random $0$ or $1$. Repeat it many times and *average*, and you get a stable, meaningful number — the expectation value. It is what experiments actually report, and the three Pauli expectations are enough to pin down a whole qubit. In `01/08` you computed the bracket $\langle\psi|A|\psi\rangle$ by hand and found the Bloch coordinates; today that bracket becomes a measurement you can run, with an error bar you can size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, born_probabilities, sample_counts, bloch_vector
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, S_DAG, apply_gate, expectation

rng = np.random.default_rng(0)   # used for the repeated-trial experiments and the exercises
viz.use_course_style()

## 1. The spectral rule: the bracket is an average of outcomes

`01/08` gave you two facts about an observable $A$ (a Hermitian matrix): its **expectation value** is the bracket $\langle A\rangle = \langle\psi|A|\psi\rangle$, and measuring $A$ returns one of its **eigenvalues** $\lambda_k$, with the Born probability $P(k) = |\langle b_k|\psi\rangle|^2$ of the matching eigenvector $|b_k\rangle$ (`01/09` made "measuring in the basis $\{|b_k\rangle\}$" precise). Here is why the two facts are one fact.

Expand the state in the eigenbasis of $A$ (orthonormal, so the basis expansion of `01/05` §3 applies): $|\psi\rangle = c_0|b_0\rangle + c_1|b_1\rangle$ with $c_k = \langle b_k|\psi\rangle$. Since $A|b_k\rangle = \lambda_k|b_k\rangle$,

$$ A|\psi\rangle = c_0\lambda_0|b_0\rangle + c_1\lambda_1|b_1\rangle, \qquad \langle\psi|A|\psi\rangle = \bar c_0 c_0\lambda_0 + \bar c_1 c_1\lambda_1 = \lambda_0\,P(0) + \lambda_1\,P(1), $$

the cross terms vanishing because $\langle b_0|b_1\rangle = 0$. This is the **spectral rule**:

$$ \langle A\rangle = \sum_k \lambda_k\,P(k) \;=\; \text{the average of the outcomes } \lambda_k, \text{ weighted by their probabilities.} $$

So the bracket is not an abstract number: it is *exactly* what you get by measuring $A$ many times and averaging the $\pm1$ (or whatever the eigenvalues are) that come out. Two cases by hand:

- **$Z$.** Eigenvectors $|0\rangle, |1\rangle$, eigenvalues $+1, -1$ (`01/08` §4): $\langle Z\rangle = (+1)P(0) + (-1)P(1) = P(0) - P(1)$ — the very formula `01/08` reached from the amplitudes, $|a_0|^2 - |a_1|^2 = z$.
- **$X$.** Eigenvectors $|\pm\rangle$, eigenvalues $\pm1$: $\langle X\rangle = P_X(+) - P_X(-)$, and `01/09` showed $P_X(\pm) = \tfrac12(1 \pm x)$, so $\langle X\rangle = x$ — again the Bloch coordinate, now reached through the *measurement* in the $X$ basis. (Same for $Y$ with $|{\pm i}\rangle$.)

One case the derivation passed over is two equal eigenvalues. On one qubit that happens only for $A = \lambda I$, where every outcome is $\lambda$. On two qubits (`01/17`) it is common: $Z\otimes I$ has each of $\pm1$ twice. The spectral rule holds unchanged there, with $P(k)$ read as the total Born probability of all the eigenvectors that share $\lambda_k$ (Nielsen & Chuang §2.2.5).

Let's check the rule numerically on a tilted state: compute the two sides separately.

In [ ]:
theta_t, phi_t = np.pi / 3, np.pi / 5            # a tilted state off every axis, used throughout
psi = qubit_state(theta=theta_t, phi=phi_t)
x, y, z = bloch_vector(psi)

# Z: eigenbasis |0>,|1>, eigenvalues +1,-1 -- spectral rule vs bracket
p = born_probabilities(psi)
print("Z: spectral rule  (+1)P(0) + (-1)P(1) =", round(p["0"] - p["1"], 4))
print("   bracket <psi|Z|psi>                =", round(expectation(psi, PAULI_Z), 4), "   Bloch z =", round(z, 4))

# X: eigenbasis |+>,|->, probabilities |<+|psi>|^2, |<-|psi>|^2 -- by hand, no rotation needed for the check
p_plus, p_minus = abs(np.vdot(KET_PLUS, psi)) ** 2, abs(np.vdot(KET_MINUS, psi)) ** 2
print("X: spectral rule  (+1)P(+) + (-1)P(-) =", round(p_plus - p_minus, 4), "   (P(+), P(-)) =", np.round((p_plus, p_minus), 4))
print("   bracket <psi|X|psi>                =", round(expectation(psi, PAULI_X), 4), "   Bloch x =", round(x, 4))
print("   via 01/09's rotate-then-measure    =", round(born_probabilities(apply_gate(HADAMARD, psi))["0"] - born_probabilities(apply_gate(HADAMARD, psi))["1"], 4))
print("\n(<X>, <Y>, <Z>) =", np.round([expectation(psi, P) for P in (PAULI_X, PAULI_Y, PAULI_Z)], 4), "  = Bloch vector", np.round((x, y, z), 4))

# The rule is general: a non-Pauli observable, A = (X + Z)/sqrt2, eigen-decomposed by numpy
A = (PAULI_X + PAULI_Z) / np.sqrt(2)
evals, evecs = np.linalg.eigh(A)                       # eigenvalues lambda_k, eigenvectors as columns
probs_k = np.abs(evecs.conj().T @ psi) ** 2           # P(k) = |<b_k|psi>|^2
print("\nA = (X+Z)/sqrt2: eigenvalues", np.round(evals, 3), "  P(k) =", np.round(probs_k, 4))
print("   spectral rule sum_k lambda_k P(k) =", round(float(evals @ probs_k), 4), "   bracket <psi|A|psi> =", round(expectation(psi, A), 4), "   (x+z)/sqrt2 =", round((x + z) / np.sqrt(2), 4))

**Read the output.** For $Z$ and for $X$, the weighted average of the eigenvalue outcomes equals the bracket to the digit, and both equal the Bloch coordinate — and for $X$ the $|+\rangle/|-\rangle$ probabilities computed directly from the overlaps agree with the rotate-then-measure route of `01/09`. The rule is not special to the Paulis: for $A = (X+Z)/\sqrt2$, NumPy's eigen-decomposition gives eigenvalues $\pm1$ and the weighted sum again equals the bracket (and equals $(x+z)/\sqrt2$, by linearity — exercise 4 makes this general). The spectral rule is what lets the lab *measure* a bracket: it says "average the outcomes", and nothing else.

## 2. Estimating $\langle Z\rangle$ from $N$ shots — and the size of the error bar

A real device never hands you $\langle Z\rangle$. It hands you $N$ outcomes, $n_0$ zeros and $n_1$ ones. The natural estimate is the **sample average of the $\pm1$ outcomes**:

$$ \widehat{\langle Z\rangle} = \frac{(+1)\,n_0 + (-1)\,n_1}{N} = \frac{n_0 - n_1}{N}. $$

Two things you can say about this estimate *before* running anything, using only that each shot is an independent draw of a variable $s = \pm1$ with $P(s = +1) = P(0)$:

- **It is unbiased.** Its average over many repetitions is $\langle s\rangle = P(0) - P(1) = \langle Z\rangle$ (the spectral rule again).
- **Its spread.** One draw has variance $\mathrm{Var}(s) = \langle s^2\rangle - \langle s\rangle^2 = 1 - \langle Z\rangle^2$ (because $s^2 = 1$ always). The average of $N$ independent draws has variance $\mathrm{Var}(s)/N$, so the estimate wobbles around the truth by a **standard error**

$$ \sigma_N = \sqrt{\frac{1 - \langle Z\rangle^2}{N}} \;\le\; \frac{1}{\sqrt N}. $$

This is **shot noise**, met here for the first time (and on a simulated device in `01/12`). The $1/\sqrt N$ rule you will see quoted everywhere is the *worst case*, reached for a fair coin ($\langle Z\rangle = 0$); a state with $\langle Z\rangle$ near $\pm1$ is cheaper to pin down, and an eigenstate ($\langle Z\rangle = \pm1$) costs a single shot — every outcome is the same, so the spread is zero. That is a prediction. Let's test it: for each $N$, run $200$ independent trials, measure the spread of the $200$ estimates, and compare with $\sigma_N$ — then watch a single run converge.

The shots come from `sample_counts(state, shots, seed)`, imported from `qot_course` in the first cell. It draws `shots` outcomes from the Born probabilities of `state` and returns the counts `{"0": n0, "1": n1}`. The `seed` fixes the draw, so a run can be repeated exactly, and the generator `rng` of the first cell hands every trial a fresh seed, so the trials are independent.

In [ ]:
exact = expectation(psi, PAULI_Z)
shot_counts = np.array([16, 64, 256, 1024, 4096, 16384])
n_trials = 200

def estimate_Z(state, shots, seed):
    '''Sample-average of the +-1 outcomes of `shots` Z-measurements: (n0 - n1)/shots.'''
    counts = sample_counts(state, shots=shots, seed=seed)
    return (counts["0"] - counts["1"]) / shots

seeds = rng.integers(0, 2**31 - 1, size=(len(shot_counts), n_trials))
trials = np.array([[estimate_Z(psi, N, int(sd)) for sd in seeds[i]] for i, N in enumerate(shot_counts)])
empirical_std = trials.std(axis=1)
predicted_std = np.sqrt((1 - exact ** 2) / shot_counts)

print(f"exact <Z> = {exact:.4f};  1 - <Z>^2 = {1 - exact**2:.4f}")
print("   N    mean of 200 estimates   empirical std   predicted sqrt((1-<Z>^2)/N)   worst case 1/sqrt(N)")
for N, m, e, pr in zip(shot_counts, trials.mean(axis=1), empirical_std, predicted_std):
    print(f"{N:6d}   {m:+.4f}                {e:.4f}          {pr:.4f}                        {1/np.sqrt(N):.4f}")

single_run = [estimate_Z(psi, N, seed=int(s)) for N, s in zip(shot_counts, rng.integers(0, 2**31 - 1, size=len(shot_counts)))]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.axhline(exact, color=COLORS["highlight"], lw=2, label=f"exact <Z> = {exact:.3f}")
ax.fill_between(shot_counts, exact - 1 / np.sqrt(shot_counts), exact + 1 / np.sqrt(shot_counts), color=COLORS["grid"], alpha=0.6, label=r"worst case $\pm 1/\sqrt{N}$")
ax.fill_between(shot_counts, exact - predicted_std, exact + predicted_std, color=COLORS["quantum"], alpha=0.3, label=r"predicted $\pm\sigma_N$")
ax.plot(shot_counts, single_run, "o-", color=COLORS["quantum"], lw=2, label="one run's estimate")
ax.set_xscale("log"); ax.set_xlabel("number of shots N"); ax.set_ylabel(r"$\widehat{\langle Z\rangle}$")
ax.set_title("One run converges toward the exact value", pad=12)
ax.legend(fontsize=8)
ax = axes[1]
ax.loglog(shot_counts, empirical_std, "o", color=COLORS["flow"], ms=8, label="empirical std of 200 trials")
ax.loglog(shot_counts, predicted_std, "-", color=COLORS["quantum"], lw=2, label=r"predicted $\sigma_N=\sqrt{(1-\langle Z\rangle^2)/N}$")
ax.loglog(shot_counts, 1 / np.sqrt(shot_counts), "--", color=COLORS["muted"], lw=1.5, label=r"worst case $1/\sqrt{N}$")
ax.set_xlabel("number of shots N"); ax.set_ylabel("spread of the estimate")
ax.set_title("The error bar shrinks as predicted", pad=12)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Read the output and the figures.** The mean of the $200$ estimates sits on the exact $\langle Z\rangle$ at every $N$ (unbiased), and their spread tracks the predicted $\sigma_N = \sqrt{(1-\langle Z\rangle^2)/N}$ — a straight line of slope $-\tfrac12$ on the log–log plot, sitting *below* the worst-case $1/\sqrt N$ by the factor $\sqrt{1 - \langle Z\rangle^2} \approx 0.87$ for this state. On the left, a single run's estimate scatters at small $N$ and settles toward the exact value as $N$ grows. Look honestly at the band: it is a $\pm1\sigma$ band, which a single estimate is *expected* to exceed about one time in three — some of the points here do, about the expected share — so "inside the band" is a statement about the spread of many runs (the right panel), not a guarantee for each one. This is the price and the promise of real measurement: more shots buy more precision, at a square-root rate, and the rate is something you can compute in advance. Every expectation value an experiment quotes should carry an error bar of this size (`01/12` runs this on a simulated device, module 05 on a real one).

## 3. Three averages, one state: the bridge to tomography

In `01/08` you saw that $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ is the Bloch vector — a complete fingerprint of a pure qubit. You now know how to *measure* all three: $\langle Z\rangle$ from a direct measurement, $\langle X\rangle$ and $\langle Y\rangle$ by first rotating those bases onto $Z$ ($H$, and $S^\dagger$ then $H$ — `01/09`), then averaging the $\pm1$ outcomes. Each estimate carries its own error bar $\sqrt{(1 - \langle P\rangle^2)/N}$ from section 2. Let's do it on the tilted state, with $N = 4000$ shots per axis, and compare with the exact Bloch vector.

This is the entire recipe of the tomography notebook `01/16`: estimate three averages, assemble the state. (`01/13` will pack the three numbers into the *density matrix* $\rho = \tfrac12\big(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z\big)$ — accept the formula as a preview today; its meaning is earned there.)

In [ ]:
N = 4000
rotations = {"X": HADAMARD, "Y": HADAMARD @ S_DAG, "Z": np.eye(2, dtype=complex)}   # rotate the basis onto Z (01/09), then measure Z
estimates, errors, exacts = {}, {}, {}
for axis, U in rotations.items():
    rotated = apply_gate(U, psi)
    estimates[axis] = estimate_Z(rotated, N, seed=int(rng.integers(0, 2**31 - 1)))
    exacts[axis] = expectation(psi, {"X": PAULI_X, "Y": PAULI_Y, "Z": PAULI_Z}[axis])
    errors[axis] = np.sqrt((1 - estimates[axis] ** 2) / N)        # plug-in error bar from the data, as a lab would (exactly 0 if every shot agrees: see below)
    print(f"<{axis}>: estimated {estimates[axis]:+.3f} +- {errors[axis]:.3f}   exact {exacts[axis]:+.3f}   off by {abs(estimates[axis] - exacts[axis]) / errors[axis]:.1f} sigma")

r_hat = np.array([estimates[a] for a in "XYZ"])
print("\nestimated Bloch vector :", np.round(r_hat, 3), "  length", round(float(np.linalg.norm(r_hat)), 3))
print("exact Bloch vector     :", np.round(bloch_vector(psi), 3), "  length 1.000 (pure state)")

fig, ax = plt.subplots(figsize=(7.0, 4.2))
labels = [r"$\langle X\rangle$", r"$\langle Y\rangle$", r"$\langle Z\rangle$"]
ax.bar(labels, [exacts[a] for a in "XYZ"], color=[COLORS["source"], COLORS["flow"], COLORS["quantum"]], alpha=0.5, label="exact (bracket)")
ax.errorbar(labels, r_hat, yerr=[errors[a] for a in "XYZ"], fmt="o", color=COLORS["highlight"], ms=8, capsize=5, label=f"estimated from {N} shots per axis")
ax.axhline(0, color=COLORS["grid"], lw=1)
ax.set_ylim(-1, 1); ax.set_ylabel("expectation value (= Bloch coordinate)")
ax.set_title("Three estimated averages reconstruct the state", pad=12)
ax.legend(fontsize=8, loc="lower left")
plt.show()

**Read the output and the figure.** Each Pauli average is recovered from shots to within its own error bar (the "off by" column counts how many $\sigma$ the estimate missed by — typically below $2$; note the error bar is itself computed from the *estimate*, as a lab must do, which at $N = 4000$ differs negligibly from the exact $\sigma_N$; the next cells show where this habit fails), and the three together give a Bloch vector that sits on top of the exact one, with length close to $1$ as a pure state requires. No new machinery was needed: the basis changes of `01/09` and the averaging of section 2. You have assembled tomography from first principles — `01/16` will run exactly this on a real device, where the length of the estimated vector will tell a story of its own.

**One trap of the plug-in bar.** The bar above is computed from the estimate, $\sqrt{(1-\widehat{\langle Z\rangle}^2)/N}$. When every shot agrees, the estimate is exactly $\pm1$ and the bar is exactly $0$. For an eigenstate that zero is right, since the exact $\sigma_N$ is zero too (exercise 2 meets this with $|{+i}\rangle$). For a state *near* a pole, measured with few shots, it is wrong: the shots happen to agree, yet the true spread is not zero. When the exact $\sigma_N$ is unknown, a standard remedy is the **Wilson score interval** (Wilson, 1927; compared with the other usual choices by Brown, Cai & DasGupta, 2001), which never shrinks to zero width. In its one-sigma version, when all $N$ shots agree, its half-width on $\langle Z\rangle$ is $1/(N+1)$. The cell below draws $200$ runs of $16$ shots on a state close to $|0\rangle$ ($\theta = 0.2$) and counts how often the plug-in bar claims no uncertainty at all.

In [ ]:
def wilson_halfwidth_Z(est, shots):
    '''Half-width of the one-sigma (z = 1) Wilson score interval for P(0), mapped to <Z> = 2 P(0) - 1.'''
    p_hat = (1 + est) / 2
    return 2 / (1 + 1 / shots) * np.sqrt(p_hat * (1 - p_hat) / shots + 1 / (4 * shots ** 2))

near_pole = qubit_state(theta=0.2, phi=0.0)
N_small, n_runs = 16, 200
exact_np = expectation(near_pole, PAULI_Z)
sigma_exact = np.sqrt((1 - exact_np ** 2) / N_small)
ests = np.array([estimate_Z(near_pole, N_small, int(s)) for s in rng.integers(0, 2**31 - 1, size=n_runs)])
plug_in = np.sqrt((1 - ests ** 2) / N_small)
all_agree = plug_in == 0

print(f"exact <Z> = {exact_np:.4f};  exact sigma_N at N = {N_small}: {sigma_exact:.4f}")
print(f"runs where all {N_small} shots agree: {all_agree.sum()} of {n_runs};  plug-in bar in those runs: {plug_in[all_agree].max():.4f}")
print(f"Wilson half-width in those runs: {wilson_halfwidth_Z(1.0, N_small):.4f}   (1/(N+1) = {1 / (N_small + 1):.4f})")
print(f"far from a pole, N = {N}, estimate 0.5:  plug-in {np.sqrt((1 - 0.5 ** 2) / N):.5f}   Wilson {wilson_halfwidth_Z(0.5, N):.5f}")

**Read the output.** In $169$ of the $200$ runs all $16$ shots agree, and the plug-in bar reads $0.0000$, while the true spread of the estimate is $\sigma_N = 0.0497$. The Wilson half-width in those runs is $0.0588 = 1/17$, of the same size as the true spread. Far from a pole, at the $N = 4000$ of section 3, the two bars agree to the printed digit ($0.01369$ for an estimate of $0.5$). So the plug-in bar is a fine tool away from the poles; near a pole or with few shots, quote the Wilson bar, or the exact $\sigma_N$ when you know the state.

## Your turn

1. **A balanced state.** Compute $\langle Z\rangle$, $\langle X\rangle$, $\langle Y\rangle$ for $|+\rangle$ by the spectral rule (which eigenbasis, which probabilities?). Then predict the error bars $\sigma_N$ of the three estimates for $N = 1000$ shots, and confirm with `estimate_Z` on the appropriately rotated state: which average is *free* (zero spread), and which is the most expensive?
2. **Estimate $\langle Y\rangle$.** Using the $S^\dagger$ then $H$ rotation of `01/09`, estimate $\langle Y\rangle$ of the tilted state `psi` from $N = 500$ shots, quote it with its error bar, and compare with the exact value. Repeat for $|{+i}\rangle$ and explain why its estimate never wobbles.
3. **Budget the shots.** You want $\langle Z\rangle$ of the tilted state to $\pm 0.01$. From $\sigma_N$, how many shots do you need? Run that many (one trial) and check the miss is of the expected size; then do the same for a fair coin — how much more does it cost?
4. **Harder: any observable, any axis.** For a unit vector $\vec n = (n_x, n_y, n_z)$ define $A = n_x X + n_y Y + n_z Z$. Show by hand, from linearity of the bracket and `01/08`, that $\langle A\rangle = \vec n\cdot\vec r$ (the Bloch vector projected on $\vec n$). Then show $A^2 = I$ using $X^2 = Y^2 = Z^2 = I$ and the anticommutation $XY = -YX$, $YZ = -ZY$, $ZX = -XZ$ (`01/08` exercise 2 proved the last one; check the other two pairs by the column rule first), so $A$'s eigenvalues are $\pm1$ — it is a "Pauli along $\vec n$". Verify on 500 random pairs $(\vec n, |\psi\rangle)$ drawn with the `rng`: `np.linalg.eigvalsh(A)` gives $\pm1$ and `expectation(psi, A)` equals $\vec n\cdot\vec r$. The error bar of section 2 therefore applies to *every* such measurement.

## What you built

- You proved the spectral rule $\langle A\rangle = \sum_k \lambda_k P(k)$ from the eigenbasis expansion, so the bracket of `01/08` *is* an average of measurement outcomes — checked by hand for $Z$ and $X$.
- You derived that the shot estimate $(n_0 - n_1)/N$ is unbiased with standard error $\sqrt{(1 - \langle Z\rangle^2)/N}$, and tested the prediction against $200$ trials per $N$ — shot noise, with $1/\sqrt N$ as its worst case.
- You estimated $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ of one state from shots, each with an error bar, and recovered its Bloch vector — the recipe at the heart of tomography (`01/16`), with the density-matrix packing previewed for `01/13`.

The abstract bracket is now a measurement you can run and an error bar you can size in advance. This is how quantum theory meets the lab bench — and you bridged them yourself.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| average $\sum_x p_x f(x)$, estimated from $N$ samples with standard error $\sqrt{\mathrm{Var}(f)/N}$ | expectation value $\langle A\rangle = \sum_k \lambda_k P(k)$, estimated from $N$ shots of a $\pm1$ observable with standard error $\sqrt{(1 - \langle A\rangle^2)/N}$ |

## What's next

You have completed the operational toolkit for one qubit: states, gates, bases, dials, and now averages with error bars. In `01/12_shot_noise_and_first_circuit` we run it as a genuine Qiskit circuit on a simulated device — and the `qc.h(0)` that opens it, the `measure` that closes it, and the shot-noise scatter in the counts are all old friends now, not mysteries.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §2.2.5 (projective measurements; average values and standard deviation), §8.4.2 (state tomography), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- J. J. Sakurai & J. Napolitano, *Modern Quantum Mechanics*, §1.4 (measurements, observables, expectation values), 3rd ed., Cambridge University Press (2020). doi:10.1017/9781108587280
- E. B. Wilson, "Probable inference, the law of succession, and statistical inference", *Journal of the American Statistical Association* **22**(158), 209–212 (1927). doi:10.1080/01621459.1927.10502953
- L. D. Brown, T. T. Cai & A. DasGupta, "Interval estimation for a binomial proportion", *Statistical Science* **16**(2), 101–133 (2001). doi:10.1214/ss/1009213286

**Previous:** `notebooks/01_QuantumFoundations/10_phase_and_rotation_gates.ipynb` · **Next:** `notebooks/01_QuantumFoundations/12_shot_noise_and_first_circuit.ipynb`